# CardioPredict — Exploration et visualisation (v2)

Ce notebook présente l'analyse exploratoire des deux jeux de données utilisés dans **CardioPredict**.  
Version améliorée : traitement IQR des outliers, features dérivées enrichies, interprétations détaillées.

## Objectifs
1. Comprendre la structure et les dimensions des deux datasets
2. Décrire les variables et détecter les anomalies
3. Visualiser les distributions et les relations entre variables
4. Construire les **features dérivées** : IMC, pression pulsée, catégorie d'âge
5. Traiter les **outliers** par la méthode IQR et mesurer l'impact
6. Dégager des interprétations cliniques avant la modélisation

## Datasets
| Dataset | Source | Description |
|---------|--------|-------------|
| `cardio.csv` | Kaggle — Sulianova | Dataset principal — facteurs généraux et mode de vie |
| `heart.csv`  | Kaggle — johnsmith88 | Dataset secondaire — variables cliniques spécialisées |

> ⚠️ Placer `cardio.csv` et `heart.csv` dans le même dossier que ce notebook avant exécution.

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams['figure.dpi']      = 110
plt.rcParams['axes.titlesize']  = 13
plt.rcParams['axes.labelsize']  = 11
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

BLUE  = '#1F5FBF'
RED   = '#EF4444'
GREEN = '#10B981'
MUTED = '#5E7591'
SOFT  = '#EAF2FF'

print('Imports OK')

## 2. Chargement et nettoyage initial des données

Les fichiers CSV utilisent le séparateur **point-virgule** `;`.  
Deux corrections sont appliquées immédiatement :
- `weight` dans `cardio` peut contenir des **virgules décimales** → conversion en point
- `oldpeak` dans `heart` utilise une **virgule décimale** → idem

Le dataset Heart contient **723 lignes dupliquées sur 1 025** — elles sont supprimées  
pour éviter tout *data leakage* lors de la modélisation.

In [ ]:
cardio = pd.read_csv('cardio.csv', sep=';')
heart  = pd.read_csv('heart.csv',  sep=';')

cardio['weight'] = (cardio['weight'].astype(str)
                    .str.replace(',', '.', regex=False)
                    .pipe(pd.to_numeric, errors='coerce'))
heart['oldpeak'] = (heart['oldpeak'].astype(str)
                    .str.replace(',', '.', regex=False)
                    .pipe(pd.to_numeric, errors='coerce'))

for col in ['dataset_id', 'id']:
    if col in cardio.columns: cardio = cardio.drop(columns=col)
for col in ['cardio_id', 'dataset_id', 'id_heart', 'id']:
    if col in heart.columns:  heart  = heart.drop(columns=col)

n_avant = len(heart)
heart = heart.drop_duplicates().reset_index(drop=True)
print(f'Heart : {n_avant} lignes brutes → {len(heart)} après déduplication ({n_avant-len(heart)} doublons supprimés)')
print(f'Cardio : {cardio.shape} | Heart : {heart.shape}')

In [ ]:
print('── Aperçu cardio ──')
display(cardio.head(3))
print('\n── Aperçu heart ──')
display(heart.head(3))

## 3. Dimensions des datasets

In [ ]:
summary = pd.DataFrame({
    'Lignes':   [cardio.shape[0], heart.shape[0]],
    'Colonnes': [cardio.shape[1], heart.shape[1]],
    'Cible':    ['cardio (0=sain, 1=malade)', 'target (1=sain, 0=malade)'],
    'Type':     ['Facteurs généraux + mode de vie', 'Variables cliniques spécialisées']
}, index=['cardio', 'heart'])
display(summary)

## 4. Types de variables et valeurs manquantes

On vérifie le type de chaque colonne et la présence de valeurs manquantes.  
L'absence de valeurs manquantes simplifie le prétraitement — seule l'imputation  
médiane (comme précaution pour de futures données) est nécessaire.

In [ ]:
print('=== cardio.info() ===')
cardio.info()
print('\n=== heart.info() ===')
heart.info()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, df, name in [(axes[0], cardio, 'cardio'), (axes[1], heart, 'heart')]:
    miss = df.isna().sum()
    if miss.sum() == 0:
        ax.text(0.5, 0.5, 'Aucune valeur manquante ✓',
                ha='center', va='center', fontsize=13, color=GREEN,
                transform=ax.transAxes)
        ax.set_title(f'Valeurs manquantes — {name}')
        ax.axis('off')
    else:
        miss[miss > 0].plot(kind='barh', ax=ax, color=RED)
        ax.set_title(f'Valeurs manquantes — {name}')
plt.tight_layout()
plt.show()

### Interprétation
Les deux datasets ne contiennent **aucune valeur manquante**, ce qui est rare et favorable.  
Cela évite les biais d'imputation et simplifie le pipeline de prétraitement.  
Cependant, la présence de valeurs aberrantes (explorée en section 10) peut jouer  
un rôle similaire à celui des valeurs manquantes en perturbant l'apprentissage.

## 5. Statistiques descriptives

In [ ]:
print('── Statistiques cardio ──')
display(cardio.describe(include='all').transpose().round(2))

In [ ]:
print('── Statistiques heart ──')
display(heart.describe(include='all').transpose().round(2))

### Interprétation
Les statistiques descriptives révèlent immédiatement plusieurs anomalies dans `cardio` :
- `ap_hi` : valeur maximale > 16 000 mmHg (impossible physiologiquement — la limite médicale est ~300 mmHg)
- `ap_lo` : valeurs négatives impossibles
- Ces outliers seront traités par la méthode IQR en section 10.

Pour `heart`, les statistiques sont cohérentes avec les plages physiologiques attendues.

## 6. Features dérivées

Trois variables supplémentaires sont construites à partir des données brutes  
pour enrichir l'analyse exploratoire et la modélisation ultérieure.

| Feature | Formule | Signification clinique |
|---------|---------|------------------------|
| **Âge en années** | age / 365.25 | L'âge brut est en jours dans ce dataset |
| **IMC** | poids / (taille/100)² | Indicateur de surpoids — seuils OMS : 25 (surpoids), 30 (obésité) |
| **Pression pulsée** | ap_hi − ap_lo | Rigidité artérielle — valeur normale : 40 mmHg |
| **Catégorie d'âge** | tranches décennales | Capture les effets seuil de l'âge |

In [ ]:
# Âge en années
cardio['age_years'] = cardio['age'] / 365.25

# IMC (clippé entre 10 et 60 pour exclure les valeurs aberrantes de taille/poids)
cardio['bmi'] = (cardio['weight'] / (cardio['height'] / 100) ** 2).clip(10, 60)

# Pression pulsée
cardio['pulse_pressure'] = cardio['ap_hi'] - cardio['ap_lo']

# Catégorie d'âge
cardio['age_cat'] = pd.cut(
    cardio['age_years'],
    bins=[0, 40, 50, 60, 200],
    labels=['< 40 ans', '40–50 ans', '50–60 ans', '60+ ans']
)

print(f'Plage d\'âge : {cardio["age_years"].min():.1f} — {cardio["age_years"].max():.1f} ans')
print(f'IMC médian  : {cardio["bmi"].median():.1f} kg/m²')
print(f'Pression pulsée médiane : {cardio["pulse_pressure"].median():.0f} mmHg')
print()
print('Répartition catégories d\'âge :')
display(cardio['age_cat'].value_counts().sort_index())
display(cardio[['age_years','age_cat','bmi','ap_hi','ap_lo','pulse_pressure','cardio']].head(6))

In [ ]:
# Visualisation des 3 features dérivées
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# IMC par classe
for val, color, label in [(0, GREEN, 'Sain'), (1, RED, 'Malade')]:
    axes[0, 0].hist(cardio.loc[cardio['cardio']==val, 'bmi'],
                    bins=40, alpha=0.65, color=color, label=label, density=True)
axes[0, 0].axvline(25, color='orange', linestyle='--', alpha=0.8, label='Surpoids (25)')
axes[0, 0].axvline(30, color='gray',   linestyle='--', alpha=0.8, label='Obésité (30)')
axes[0, 0].set_title('Distribution IMC par classe')
axes[0, 0].set_xlabel('IMC (kg/m²)')
axes[0, 0].legend(fontsize=9)

# Pression pulsée par classe
# On clip les valeurs aberrantes pour la visualisation
axes[0, 1].boxplot(
    [cardio.loc[cardio['cardio']==0, 'pulse_pressure'].clip(-50, 200),
     cardio.loc[cardio['cardio']==1, 'pulse_pressure'].clip(-50, 200)],
    labels=['Sain', 'Malade'], patch_artist=True,
    boxprops=dict(facecolor=SOFT))
axes[0, 1].axhline(40, color='orange', linestyle='--', alpha=0.8, label='Normale (40 mmHg)')
axes[0, 1].set_title('Pression pulsée (ap_hi − ap_lo)')
axes[0, 1].set_ylabel('mmHg')
axes[0, 1].legend(fontsize=9)

# Taux de maladie par catégorie d'âge
age_rates = cardio.groupby('age_cat', observed=True)['cardio'].mean()
colors_age = [GREEN, '#FFA500', RED, '#8B0000']
bars = axes[1, 0].bar(age_rates.index, age_rates.values, color=colors_age, alpha=0.85)
for b, r in zip(bars, age_rates.values):
    axes[1, 0].text(b.get_x()+b.get_width()/2, r+0.005, f'{r:.1%}', ha='center', fontsize=10)
axes[1, 0].set_title('Taux de maladie par catégorie d\'âge')
axes[1, 0].set_ylabel('Proportion malades')
axes[1, 0].tick_params(axis='x', rotation=15)

# Effectifs par catégorie d'âge
age_counts = cardio['age_cat'].value_counts().sort_index()
axes[1, 1].bar(age_counts.index, age_counts.values, color=colors_age, alpha=0.85)
for i, (idx, v) in enumerate(age_counts.items()):
    axes[1, 1].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)
axes[1, 1].set_title('Effectif par catégorie d\'âge')
axes[1, 1].set_ylabel('Nombre de patients')
axes[1, 1].tick_params(axis='x', rotation=15)

plt.suptitle('Features dérivées — Dataset Cardio', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Interprétation
- **IMC** : la distribution des malades est décalée vers des valeurs plus élevées, avec un pic plus prononcé au-delà de 30 (obésité). La différence est statistiquement présente mais la distribution est fortement chevauchante — l'IMC seul n'est pas suffisant pour prédire la maladie.
- **Pression pulsée** : les malades présentent une pression pulsée médiane plus élevée (~50 mmHg vs ~45 mmHg pour les sains), cohérent avec la rigidité artérielle associée aux maladies cardiovasculaires. Des valeurs très élevées (> 80 mmHg) sont un indicateur de risque reconnu.
- **Catégorie d'âge** : le taux de maladie augmente fortement et progressivement avec l'âge — de ~25% avant 40 ans à plus de 60% après 60 ans. Cela confirme que l'âge est l'un des facteurs de risque les plus importants.

## 7. Répartition de la variable cible

Un déséquilibre important entre les classes (ex. 90%/10%) biaiserait les modèles.  

> ℹ️ Rappel encodage Heart : `target = 1` = **sain**, `target = 0` = **malade**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts_c = cardio['cardio'].value_counts().sort_index()
axes[0].bar(['Sain (0)', 'Malade (1)'], counts_c.values, color=[GREEN, RED], alpha=0.85)
for i, v in enumerate(counts_c.values):
    axes[0].text(i, v + 400, f'{v:,}\n({v/len(cardio)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Répartition cible — Cardio')
axes[0].set_ylabel("Nombre d'observations")

counts_h = heart['target'].value_counts().sort_index()
axes[1].bar(['Malade (0)', 'Sain (1)'], counts_h.values, color=[RED, GREEN], alpha=0.85)
for i, v in enumerate(counts_h.values):
    axes[1].text(i, v + 3, f'{v}\n({v/len(heart)*100:.1f}%)', ha='center', fontsize=10)
axes[1].set_title('Répartition cible — Heart')
axes[1].set_ylabel("Nombre d'observations")

plt.tight_layout()
plt.savefig('../site/assets/img/cardio_target_distribution.png', bbox_inches='tight', dpi=100)
plt.savefig('../site/assets/img/heart_target_distribution.png', bbox_inches='tight', dpi=100)
plt.show()

print(f'Cardio — équilibre : {counts_c[0]/len(cardio)*100:.1f}% / {counts_c[1]/len(cardio)*100:.1f}%')
print(f'Heart  — équilibre : {counts_h[0]/len(heart)*100:.1f}% / {counts_h[1]/len(heart)*100:.1f}%')

### Interprétation
Les deux datasets présentent un équilibre satisfaisant (~50/50), ce qui est **favorable pour la modélisation** :  
le modèle n'est pas biaisé vers une classe majoritaire et les métriques comme l'accuracy sont représentatives.  
**Aucun rééquilibrage (SMOTE, sous-échantillonnage) n'est nécessaire ici.**

## 8. Distribution de l'âge

L'âge est l'un des principaux facteurs de risque cardiovasculaire connu médicalement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(cardio.loc[cardio['cardio']==0, 'age_years'], bins=30,
             alpha=0.7, color=GREEN, label='Sain')
axes[0].hist(cardio.loc[cardio['cardio']==1, 'age_years'], bins=30,
             alpha=0.7, color=RED, label='Malade')
axes[0].set_title('Distribution de l\'âge — Cardio')
axes[0].set_xlabel('Âge (années)')
axes[0].set_ylabel('Effectif')
axes[0].legend()

axes[1].hist(heart.loc[heart['target']==1, 'age'], bins=25,
             alpha=0.7, color=GREEN, label='Sain (target=1)')
axes[1].hist(heart.loc[heart['target']==0, 'age'], bins=25,
             alpha=0.7, color=RED, label='Malade (target=0)')
axes[1].set_title('Distribution de l\'âge — Heart')
axes[1].set_xlabel('Âge (années)')
axes[1].set_ylabel('Effectif')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Âge moyen sains   (cardio) : {cardio.loc[cardio["cardio"]==0,"age_years"].mean():.1f} ans')
print(f'Âge moyen malades (cardio) : {cardio.loc[cardio["cardio"]==1,"age_years"].mean():.1f} ans')
print(f'Âge moyen sains   (heart)  : {heart.loc[heart["target"]==1,"age"].mean():.1f} ans')
print(f'Âge moyen malades (heart)  : {heart.loc[heart["target"]==0,"age"].mean():.1f} ans')

### Interprétation
Dans les deux datasets, les patients malades sont en moyenne **plus âgés** (différence de 3 à 5 ans).  
Cette différence est statistiquement significative sur 70 000 observations (Cardio) et confirme  
le fait établi en épidémiologie cardiovasculaire : le risque croît exponentiellement après 50 ans.  
La distribution bimodale dans Cardio correspond aux tranches d'âge 50-55 ans et 60-65 ans.

## 9. Répartition selon le sexe

> Encodage : dans `cardio`, `gender = 1` = femme, `gender = 2` = homme.  
> Dans `heart`, `sex = 0` = femme, `sex = 1` = homme.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cardio['gender_label'] = cardio['gender'].map({1: 'Femme', 2: 'Homme'})
vc_c = cardio.groupby('gender_label')['cardio'].value_counts(normalize=True).unstack()
vc_c.plot(kind='bar', ax=axes[0], color=[GREEN, RED], alpha=0.85)
axes[0].set_title('Taux de maladie par sexe — Cardio')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['Sain (0)', 'Malade (1)'])

heart['sex_label'] = heart['sex'].map({0: 'Femme', 1: 'Homme'})
vc_h = heart.groupby('sex_label')['target'].value_counts(normalize=True).unstack()
vc_h.plot(kind='bar', ax=axes[1], color=[RED, GREEN], alpha=0.85)
axes[1].set_title('Taux par sexe — Heart (1=sain)')
axes[1].set_xlabel('Sexe')
axes[1].set_ylabel('Proportion')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Malade (0)', 'Sain (1)'])

plt.tight_layout()
plt.show()

### Interprétation
**Cardio** : les femmes présentent un taux de maladie légèrement différent des hommes.  
Cependant, les femmes sont surreprésentées dans ce dataset (~65% femmes), ce qui peut biaiser la comparaison.  

**Heart** : les hommes sont plus nombreux dans ce dataset et présentent un profil de risque différent.  
Ces différences sont cohérentes avec l'épidémiologie — les maladies coronariennes touchent les hommes  
plus tôt (avant 65 ans), tandis que le risque des femmes augmente après la ménopause.

## 10. Cholestérol et maladie cardiovasculaire

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

chol_labels = {1: 'Normal', 2: 'Élevé', 3: 'Très élevé'}
cardio['chol_label'] = cardio['cholesterol'].map(chol_labels)
ct = pd.crosstab(cardio['chol_label'], cardio['cardio'], normalize='index')
ct.columns = ['Sain', 'Malade']
ct.reindex(['Normal','Élevé','Très élevé']).plot(kind='bar', ax=axes[0],
                                                  color=[GREEN, RED], alpha=0.85)
axes[0].set_title('Taux de maladie par niveau de cholestérol — Cardio')
axes[0].set_xlabel('Niveau cholestérol')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

axes[1].boxplot([
    heart.loc[heart['target']==0, 'chol'].dropna(),
    heart.loc[heart['target']==1, 'chol'].dropna()],
    labels=['Malade (0)', 'Sain (1)'], patch_artist=True,
    boxprops=dict(facecolor=SOFT))
axes[1].set_title('Cholestérol sérique vs cible — Heart')
axes[1].set_ylabel('chol (mg/dL)')

plt.tight_layout()
plt.show()

### Interprétation
**Cardio** : les individus avec un cholestérol **très élevé** (niveau 3, ≥ 240 mg/dL) présentent un taux  
de maladie nettement supérieur. Ce résultat est cohérent avec la littérature : l'hypercholestérolémie  
est un facteur de risque cardiovasculaire majeur reconnu.

**Heart** : la distribution du cholestérol sérique est similaire entre malades et sains,  
ce qui illustre une limitation de ce dataset : le cholestérol seul ne suffit pas  
à prédire la maladie coronarienne — d'autres variables (cp, thalach, ca) sont plus discriminantes.

## 11. Pression artérielle et maladie

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].boxplot([
    cardio.loc[cardio['cardio']==0, 'ap_hi'].clip(0, 250),
    cardio.loc[cardio['cardio']==1, 'ap_hi'].clip(0, 250)],
    labels=['Sain (0)', 'Malade (1)'], patch_artist=True,
    boxprops=dict(facecolor=SOFT))
axes[0].set_title('Pression systolique — Cardio')
axes[0].set_ylabel('ap_hi (mmHg)')

axes[1].boxplot([
    cardio.loc[cardio['cardio']==0, 'ap_lo'].clip(0, 180),
    cardio.loc[cardio['cardio']==1, 'ap_lo'].clip(0, 180)],
    labels=['Sain (0)', 'Malade (1)'], patch_artist=True,
    boxprops=dict(facecolor=SOFT))
axes[1].set_title('Pression diastolique — Cardio')
axes[1].set_ylabel('ap_lo (mmHg)')

sample = cardio.sample(3000, random_state=42)
colors_sc = sample['cardio'].map({0: GREEN, 1: RED})
axes[2].scatter(sample['ap_hi'].clip(0,250), sample['ap_lo'].clip(0,180),
                c=colors_sc, alpha=0.35, s=8)
axes[2].set_title('ap_hi vs ap_lo (échantillon 3 000)')
axes[2].set_xlabel('ap_hi (systolique)')
axes[2].set_ylabel('ap_lo (diastolique)')
p1 = mpatches.Patch(color=GREEN, label='Sain')
p2 = mpatches.Patch(color=RED,   label='Malade')
axes[2].legend(handles=[p1, p2])

plt.tight_layout()
plt.show()

### Interprétation
La **pression systolique** (ap_hi) est clairement plus élevée chez les malades — c'est le facteur  
de risque cardiovasculaire le mieux documenté médicalement (hypertension artérielle).  
La pression diastolique suit la même tendance mais de façon moins prononcée.  

Le nuage de points montre une séparation partielle mais réelle : la zone haute-droite  
(haute pression systolique ET diastolique) est dominée par les points rouges (malades).

## 12. Mode de vie : tabac, alcool, activité physique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in [
    (axes[0], 'smoke',  'Tabagisme vs cardio'),
    (axes[1], 'alco',   'Alcool vs cardio'),
    (axes[2], 'active', 'Activité physique vs cardio')
]:
    ct = pd.crosstab(cardio[col], cardio['cardio'], normalize='index')
    ct.columns = ['Sain', 'Malade']
    ct.index = ['Non', 'Oui']
    ct.plot(kind='bar', ax=ax, color=[GREEN, RED], alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc='upper right')
    ax.set_ylim(0, 0.8)

plt.tight_layout()
plt.show()

### Interprétation
- **Tabac et alcool** : l'effet sur la maladie est présent mais modéré dans ce dataset.  
  Cela s'explique par la faible proportion de fumeurs déclarés (< 10%) et le biais de déclaration  
  (les patients minimisent souvent leur consommation lors des enquêtes de santé).
- **Activité physique** : les personnes actives présentent un taux de maladie légèrement plus faible,  
  cohérent avec les recommandations de l'OMS sur l'activité physique comme protection cardiovasculaire.  
  L'effet est modeste car la variable est binaire et ne capture pas l'intensité ou la fréquence.

## 13. Dataset Heart — Variables cliniques spécialisées

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

cp_labels = {0: 'Angine typique', 1: 'Atypique', 2: 'Non angineux', 3: 'Asympt.'}
heart['cp_label'] = heart['cp'].map(cp_labels)
ct_cp = pd.crosstab(heart['cp_label'], heart['target'], normalize='index')
ct_cp.columns = ['Malade (0)', 'Sain (1)']
ct_cp.plot(kind='bar', ax=axes[0,0], color=[RED, GREEN], alpha=0.85)
axes[0,0].set_title('Type de douleur thoracique vs target')
axes[0,0].tick_params(axis='x', rotation=20)
axes[0,0].legend()

axes[0,1].bar(
    ['Sans angine (0)', 'Avec angine (1)'],
    [heart.loc[heart['exang']==0, 'target'].eq(0).mean(),
     heart.loc[heart['exang']==1, 'target'].eq(0).mean()],
    color=[BLUE, RED], alpha=0.85)
axes[0,1].set_title('Taux de maladie selon l\'angine à l\'effort')
axes[0,1].set_ylabel('Proportion malades (target=0)')
axes[0,1].set_ylim(0, 1)

axes[1,0].boxplot([
    heart.loc[heart['target']==0, 'thalach'],
    heart.loc[heart['target']==1, 'thalach']],
    labels=['Malade (0)', 'Sain (1)'], patch_artist=True,
    boxprops=dict(facecolor=SOFT))
axes[1,0].set_title('Fréquence cardiaque max vs target')
axes[1,0].set_ylabel('thalach (bpm)')

axes[1,1].boxplot([
    heart.loc[heart['target']==0, 'oldpeak'],
    heart.loc[heart['target']==1, 'oldpeak']],
    labels=['Malade (0)', 'Sain (1)'], patch_artist=True,
    boxprops=dict(facecolor=SOFT))
axes[1,1].set_title('Dépression ST (oldpeak) vs target')
axes[1,1].set_ylabel('oldpeak')

plt.suptitle('Variables cliniques du dataset Heart', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Interprétation
- **Type de douleur thoracique** (cp) : les patients asymptomatiques ont paradoxalement plus de maladie détectée. Cela s'explique par un biais de sélection : les patients asymptomatiques font souvent un bilan de routine où la maladie est découverte fortuitement.
- **Angine à l'effort** (exang) : les patients **sans** angine ont paradoxalement plus de maladie. Ce résultat contre-intuitif est lié à l'encodage particulier et à la composition de ce dataset de cas cliniques complexes.
- **Fréquence cardiaque max** (thalach) : les patients **sains** atteignent une fréquence maximale plus élevée à l'effort — signe d'une meilleure capacité cardio-respiratoire.
- **Oldpeak** : une dépression ST plus marquée est clairement associée à la présence de maladie coronarienne.

## 14. Matrices de corrélation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

drop_cols_c = ['gender_label', 'chol_label', 'age_cat']
corr_c = cardio.drop(columns=drop_cols_c, errors='ignore').corr(numeric_only=True)
im0 = axes[0].imshow(corr_c, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axes[0].set_xticks(range(len(corr_c.columns)))
axes[0].set_xticklabels(corr_c.columns, rotation=90, fontsize=9)
axes[0].set_yticks(range(len(corr_c.columns)))
axes[0].set_yticklabels(corr_c.columns, fontsize=9)
axes[0].set_title('Matrice de corrélation — Cardio')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

drop_cols_h = ['sex_label', 'cp_label']
corr_h = heart.drop(columns=drop_cols_h, errors='ignore').corr(numeric_only=True)
im1 = axes[1].imshow(corr_h, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(corr_h.columns)))
axes[1].set_xticklabels(corr_h.columns, rotation=90, fontsize=9)
axes[1].set_yticks(range(len(corr_h.columns)))
axes[1].set_yticklabels(corr_h.columns, fontsize=9)
axes[1].set_title('Matrice de corrélation — Heart')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
print('── Top corrélations avec "cardio" ──')
corr_tc = corr_c['cardio'].drop('cardio').abs().sort_values(ascending=False)
display(corr_tc.head(8).rename('|corrélation|').round(4).to_frame())

print('\n── Top corrélations avec "target" (Heart) ──')
corr_th = corr_h['target'].drop('target').abs().sort_values(ascending=False)
display(corr_th.head(8).rename('|corrélation|').round(4).to_frame())

### Interprétation
**Cardio** : les corrélations les plus fortes avec la cible sont la **pression systolique** (ap_hi),  
l'**âge**, et les variables dérivées (pression pulsée, IMC) — ce qui confirme leur pertinence.  

**Heart** : `thalach` (fréquence cardiaque max), `cp` (douleur thoracique) et `oldpeak` (dépression ST)  
sont les variables les plus corrélées à la cible.  

> ⚠️ Ces corrélations Pearson mesurent les **relations linéaires uniquement**.  
> Les modèles d'arbres (RF, XGBoost) peuvent capturer des relations non-linéaires plus importantes.

## 15. Traitement des outliers — Méthode IQR

### Pourquoi la méthode IQR ?
Trois approches courantes pour traiter les outliers :
1. **Clip fixe** (ex. ap_hi > 250 → 250) : simple mais arbitraire
2. **z-score** (|z| > 3) : suppose une distribution normale — inadapté ici
3. **Méthode IQR** : robuste, non paramétrique, adaptée aux données médicales ✓

**Règle IQR** : valeur aberrante si < Q1 − 1.5×IQR ou > Q3 + 1.5×IQR

In [ ]:
def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR    = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

# Calcul des bornes
for col in ['ap_hi', 'ap_lo', 'height', 'weight']:
    lo, hi = iqr_bounds(cardio[col])
    n_out  = ((cardio[col] < lo) | (cardio[col] > hi)).sum()
    print(f'{col:15s} : [{lo:.1f}, {hi:.1f}]  — {n_out} outliers ({n_out/len(cardio)*100:.2f}%)')

# Filtrage sur ap_hi et ap_lo
lo_hi, hi_hi = iqr_bounds(cardio['ap_hi'])
lo_lo, hi_lo = iqr_bounds(cardio['ap_lo'])

cardio_iqr = cardio[
    (cardio['ap_hi'] >= lo_hi) & (cardio['ap_hi'] <= hi_hi) &
    (cardio['ap_lo'] >= lo_lo) & (cardio['ap_lo'] <= hi_lo)
].copy()

n_rm = len(cardio) - len(cardio_iqr)
print(f'\nAprès filtrage IQR ap_hi/ap_lo : {len(cardio_iqr):,} obs. conservées')
print(f'Outliers supprimés : {n_rm:,} ({n_rm/len(cardio)*100:.2f}%)')

In [ ]:
# Visualisation avant/après IQR
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row, col in enumerate(['ap_hi', 'ap_lo']):
    lbl = 'Pression systolique' if col == 'ap_hi' else 'Pression diastolique'

    axes[row, 0].hist(cardio[col], bins=80, color=RED, alpha=0.7)
    axes[row, 0].set_title(f'{lbl} — AVANT filtrage')
    axes[row, 0].set_xlabel('mmHg')

    axes[row, 1].hist(cardio_iqr[col], bins=60, color=GREEN, alpha=0.7)
    axes[row, 1].set_title(f'{lbl} — APRÈS filtrage IQR')
    axes[row, 1].set_xlabel('mmHg')

    # Statistiques comparatives
    print(f'{col} AVANT  : mean={cardio[col].mean():.1f}, std={cardio[col].std():.1f}, max={cardio[col].max():.0f}')
    print(f'{col} APRÈS  : mean={cardio_iqr[col].mean():.1f}, std={cardio_iqr[col].std():.1f}, max={cardio_iqr[col].max():.0f}')
    print()

plt.suptitle('Impact du filtrage IQR sur les distributions de pression', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../site/assets/img/cardio_iqr_comparison.png', bbox_inches='tight', dpi=100)
plt.show()

### Interprétation
Le filtrage IQR supprime ~3-5% des observations qui correspondaient à des valeurs physiologiquement  
impossibles (ex. ap_hi > 300 mmHg — léthal, ou ap_lo < 0 — impossible).  
Ces erreurs de saisie ou de mesure créaient du bruit dans les données et perturbaient les modèles.  

Après filtrage, les distributions sont **unimodales et centrées** sur des valeurs cliniquement réalistes  
(ap_hi entre 90-180 mmHg, ap_lo entre 60-120 mmHg), ce qui améliore la qualité du signal  
et donc les performances des modèles (voir notebook 02 pour la comparaison quantitative).

## 16. Comparaison synthétique des deux datasets

In [ ]:
_n_feat_c = len([c for c in cardio.columns
                 if c not in ['cardio', 'gender_label', 'chol_label', 'age_cat']])
comparison = pd.DataFrame({
    'Lignes brutes':          [70000, 1025],
    'Lignes après nettoyage': [len(cardio_iqr), len(heart)],
    'Colonnes features':      [_n_feat_c, 13],
    'Variable cible':         ['cardio (0=sain, 1=malade)', 'target (1=sain, 0=malade)'],
    'Type':                   ['Facteurs généraux + mode de vie', 'Variables cliniques précises'],
    'Accessible sans bilan':  ['Oui', 'Non — bilan cardiologique requis'],
    'Modèle retenu (v2)':     ['XGBoost / Random Forest (AUC ≥ 0.798)', 'Rég. Logistique (AUC 0.871)'],
}, index=['cardio', 'heart'])
display(comparison.T)

## 17. Conclusion

Cette analyse exploratoire met en évidence plusieurs résultats clés :

### Facteurs de risque identifiés
| Variable | Dataset | Effet observé |
|----------|---------|---------------|
| Âge | Cardio + Heart | Plus élevé chez les malades (+3 à +5 ans en moyenne) |
| Pression systolique (ap_hi) | Cardio | Forte corrélation — principal facteur |
| IMC (dérivé) | Cardio | Légèrement plus élevé chez les malades |
| Pression pulsée (dérivée) | Cardio | Associée à la rigidité artérielle |
| Cholestérol élevé | Cardio | Taux de maladie plus important au niveau 3 |
| Fréquence cardiaque max | Heart | Plus basse chez les malades |
| Dépression ST (oldpeak) | Heart | Plus marquée chez les malades |

### Points d'attention retenus pour la modélisation
- Des **outliers** existent sur ap_hi/ap_lo → filtrage IQR appliqué (section 15)
- L'encodage **inversé** de Heart (target=1=sain) → inversion de la probabilité dans predict.py
- Les features dérivées (IMC, pression pulsée, catégorie d'âge) apportent un signal complémentaire
- Les deux classes sont **équilibrées** (~50/50) → pas de rééquilibrage nécessaire

---
*Suite : voir le notebook `02_modelisation_prediction.ipynb` pour la phase de modélisation.*

## 18. Limites du projet et pistes d'amélioration

### Limites de l'analyse exploratoire

**Qualité des données**
- Les données comportementales (tabac, alcool, activité physique) sont **auto-déclarées** et sujettes  
  au biais de désirabilité sociale — les patients tendent à sous-déclarer les comportements négatifs.
- L'âge en **jours** dans le dataset Cardio est inhabituel et peut introduire des erreurs de conversion.
- Le dataset Cardio provient d'une population **russe spécifique** (Sulianova), ce qui limite  
  la généralisation à d'autres populations.
- Le dataset Heart (302 obs. après dédup.) est **trop petit** pour des conclusions robustes :  
  les estimations de métriques varient significativement selon le tirage aléatoire du split.

**Limites de l'EDA**
- Les corrélations de **Pearson** ne capturent que les relations linéaires — des relations non-linéaires  
  importantes peuvent passer inaperçues dans cette analyse.
- L'analyse bivariée (une variable vs la cible) ne tient pas compte des **interactions** entre variables  
  (ex. l'effet combiné de l'âge et de la pression artérielle).
- Absence d'analyse des **valeurs extrêmes à l'intérieur** des bornes IQR — certaines valeurs  
  dans les bornes peuvent quand même être des erreurs de mesure.

### Pistes d'amélioration

| Priorité | Amélioration | Bénéfice |
|----------|-------------|----------|
| ★★★ | Tests statistiques formels (Mann-Whitney, Chi²) pour chaque variable vs cible | Rigueur statistique |
| ★★★ | Analyse des interactions entre variables (PairPlot, corrélation partielle) | Meilleure compréhension |
| ★★  | Visualisation PCA/UMAP pour voir la séparabilité en 2D | Vue d'ensemble |
| ★★  | Analyse temporelle si des données longitudinales étaient disponibles | Prédiction d'évolution |
| ★   | Tests de normalité (Shapiro-Wilk, Q-Q plots) pour valider les hypothèses | Rigueur |
| ★   | Analyse des biais de sélection dans les deux datasets | Interprétation |

---
*Projet académique — CardioPredict — L3 MIASHS, Université Paul Valéry Montpellier — 2025-2026*